# All-Targets Sweep — Results Analysis

Reads `summary.json` from all 48 evaluated targets and produces:
1. **Ranked bar chart** — all targets by Nuc Median Pearson r, coloured by binary group (cell cycle vs signaling)
2. **Group comparison** — strip plot comparing cell cycle vs non-cell-cycle distributions
3. **SSIM vs Pearson scatter** — reveals decoupling between pixel-level and cell-level metrics
4. **Text summary** — tier breakdown + notable cases

Figures saved to `local_runs/all_targets_*.png`.

**Binary classification:**
- `cell_cycle`: proteins directly in the CDK/cyclin/CKI/Rb–E2F/APC-C/checkpoint machinery + DNA content + cMyc (28 targets)
- `non_cell_cycle`: signaling pathway proteins whose variation is driven by extracellular stimuli not encoded in the 7 input markers (20 targets)

**Hypothesis**: cell cycle proteins should be more predictable from the 7 cell cycle marker inputs; low Pearson for signaling proteins is expected.

In [ ]:
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LOCAL_BASE = Path("../local_runs/all_targets").resolve()

# Binary classification: cell cycle regulators vs signaling / non-cell-cycle
# Cell cycle: proteins directly in the CDK/cyclin/CKI/Rb-E2F/APC-C/checkpoint machinery
#             + DNA content channels + cMyc (direct E2F target, tightly cell-cycle coupled)
# Non-cell cycle: signaling pathway proteins whose variation is driven by
#                 extracellular stimuli not encoded in the 7 input markers
CELL_CYCLE_TARGETS = {
    # CDKs
    "CDK2", "CDK4", "CDK6",
    # Cyclin (D1 is the only cyclin target; cycA/B1/E are inputs)
    "D1",
    # CKIs & tumour suppressors
    "p16", "p21", "p27", "p53", "ARF",
    # APC/C & SCF regulators
    "Cdh1", "Cdt1", "Skp2",
    # Rb/E2F axis
    "E2F1",
    # DNA damage / checkpoint
    "pChk1", "pH2AX", "p38", "pp38",
    # Phospho-forms of cell cycle proteins
    "pp21", "pp27", "pp53",
    # DNA content (directly reflects cell cycle phase)
    "DNA_2", "DNA_3", "DNA_4", "DNA_5", "DNA_6", "DNA_7", "DNA_8",
    # cMyc: direct E2F transcriptional target, tightly cell-cycle coupled
    "cMyc",
}

GROUP_COLORS = {
    "cell_cycle":     "#4C72B0",
    "non_cell_cycle": "#C44E52",
}
GROUP_LABELS = {
    "cell_cycle":     "Cell cycle",
    "non_cell_cycle": "Signaling / other",
}

# Discover all run dirs dynamically (each is <target>_<jobid>)
rows = []
missing = []
for run_dir in sorted(LOCAL_BASE.iterdir()):
    if not run_dir.is_dir():
        continue
    summary_path = run_dir / "summary.json"
    if not summary_path.exists():
        missing.append(run_dir.name)
        continue
    with summary_path.open() as f:
        s = json.load(f)

    target = s.get("target_name", run_dir.name)
    group = "cell_cycle" if target in CELL_CYCLE_TARGETS else "non_cell_cycle"

    m  = s.get("metrics", {})
    cc = s.get("cell_correlations", {})

    rows.append({
        "target":              target,
        "group":               group,
        "ssim_mean":           m.get("val_ssim", {}).get("mean"),
        "ssim_median":         m.get("val_ssim", {}).get("median"),
        "ssim_std":            m.get("val_ssim", {}).get("std"),
        "l1_mean":             m.get("val_l1",   {}).get("mean"),
        "l1_median":           m.get("val_l1",   {}).get("median"),
        "nuc_median_pearson":  cc.get("nuc_median", {}).get("pearson"),
        "nuc_median_spearman": cc.get("nuc_median", {}).get("spearman"),
        "cyto_median_pearson": cc.get("cyto_median", {}).get("pearson"),
        "n":                   cc.get("nuc_median", {}).get("n"),
    })

if missing:
    print(f"[MISSING summary.json] {missing}")

df = pd.DataFrame(rows).sort_values("nuc_median_pearson", ascending=False).reset_index(drop=True)
print(f"Loaded {len(df)} targets  |  cell_cycle: {(df.group=='cell_cycle').sum()}  non_cell_cycle: {(df.group=='non_cell_cycle').sum()}")
df[["target", "group", "nuc_median_pearson", "nuc_median_spearman", "ssim_mean", "l1_mean"]]

In [ ]:
# --- Ranked bar chart: all 48 targets by Nuc Median Pearson ---
fig, ax = plt.subplots(figsize=(18, 5))

colors = [GROUP_COLORS[g] for g in df["group"]]
ax.bar(df["target"], df["nuc_median_pearson"], color=colors)

ax.axhline(0.9, color="green",  linestyle="--", linewidth=1.2)
ax.axhline(0.8, color="orange", linestyle="--", linewidth=1.2)
ax.axhline(0.7, color="red",    linestyle=":",  linewidth=1.0)
ax.text(len(df) - 0.5, 0.905, "r = 0.9", va="bottom", ha="right", color="green",  fontsize=8)
ax.text(len(df) - 0.5, 0.805, "r = 0.8", va="bottom", ha="right", color="orange", fontsize=8)
ax.text(len(df) - 0.5, 0.705, "r = 0.7", va="bottom", ha="right", color="red",    fontsize=8)

ax.set_ylabel("Nuc Median Pearson r")
ax.set_xlabel("Target")
ax.set_title("StainNet all-targets sweep — ranked by cell-level Pearson r (test set, n=865)")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha="right", fontsize=8)

patches = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g]) for g in GROUP_COLORS]
ax.legend(handles=patches, loc="lower left", fontsize=9)

plt.tight_layout()
plt.savefig(LOCAL_BASE.parent / "all_targets_ranked_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Group comparison: cell cycle vs non-cell-cycle strip plot ---
rng = np.random.default_rng(42)
group_order = ["cell_cycle", "non_cell_cycle"]

fig, ax = plt.subplots(figsize=(7, 6))

for i, g in enumerate(group_order):
    sub = df[df["group"] == g]
    vals = sub["nuc_median_pearson"].values
    names = sub["target"].values
    jitter = rng.uniform(-0.15, 0.15, len(vals))
    xs = i + jitter
    ax.scatter(xs, vals, color=GROUP_COLORS[g], s=70, zorder=3, alpha=0.85)
    for x, v, name in zip(xs, vals, names):
        ax.annotate(name, (x, v), textcoords="offset points", xytext=(6, 1), fontsize=7)
    mean_val = np.mean(vals)
    ax.hlines(mean_val, i - 0.25, i + 0.25, colors=GROUP_COLORS[g], linewidth=2.5,
              label=f"{GROUP_LABELS[g]} (n={len(vals)}, mean={mean_val:.3f})")

ax.axhline(0.9, color="green",  linestyle="--", linewidth=1.0, alpha=0.6)
ax.axhline(0.8, color="orange", linestyle="--", linewidth=1.0, alpha=0.6)
ax.set_xticks([0, 1])
ax.set_xticklabels([GROUP_LABELS[g] for g in group_order], fontsize=11)
ax.set_ylabel("Nuc Median Pearson r")
ax.set_title("Cell-level Pearson: cell cycle regulators vs signaling proteins\n(horizontal bar = group mean)")
ax.set_ylim(0.3, 1.05)
ax.legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(LOCAL_BASE.parent / "all_targets_group_strip.png", dpi=150, bbox_inches="tight")
plt.show()

# Descriptive stats per group
print(f"{'Group':<22} {'n':>3}  {'mean':>6}  {'median':>7}  {'min':>5}  {'max':>5}")
print("-" * 55)
for g in group_order:
    vals = df[df["group"] == g]["nuc_median_pearson"].values
    print(f"{GROUP_LABELS[g]:<22} {len(vals):>3}  {np.mean(vals):>6.3f}  {np.median(vals):>7.3f}  {np.min(vals):>5.3f}  {np.max(vals):>5.3f}")

In [ ]:
# --- SSIM vs Pearson scatter: reveals pixel/cell-level decoupling ---
fig, ax = plt.subplots(figsize=(9, 7))

for _, row in df.iterrows():
    ax.scatter(
        row["ssim_mean"], row["nuc_median_pearson"],
        color=GROUP_COLORS[row["group"]], s=70, zorder=3, alpha=0.85,
    )
    ax.annotate(
        row["target"],
        (row["ssim_mean"], row["nuc_median_pearson"]),
        textcoords="offset points", xytext=(5, 3), fontsize=7,
    )

ax.axhline(0.9, color="green",  linestyle="--", linewidth=1.0, alpha=0.6)
ax.axhline(0.8, color="orange", linestyle="--", linewidth=1.0, alpha=0.6)
ax.axvline(0.6, color="gray",   linestyle=":",  linewidth=1.0, alpha=0.6)
ax.text(0.61, 0.32, "SSIM = 0.6", color="gray", fontsize=8)

ax.set_xlabel("SSIM mean  (pixel-level reconstruction quality)")
ax.set_ylabel("Nuc Median Pearson r  (cell-level expression ranking)")
ax.set_title("SSIM vs Pearson r — decoupling of pixel and cell-level metrics")

patches = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g]) for g in GROUP_COLORS]
ax.legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig(LOCAL_BASE.parent / "all_targets_ssim_vs_pearson.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Text summary ---
strong   = df[df["nuc_median_pearson"] >= 0.9]
good     = df[(df["nuc_median_pearson"] >= 0.8) & (df["nuc_median_pearson"] < 0.9)]
moderate = df[(df["nuc_median_pearson"] >= 0.7) & (df["nuc_median_pearson"] < 0.8)]
weak     = df[(df["nuc_median_pearson"] >= 0.5) & (df["nuc_median_pearson"] < 0.7)]
failed   = df[df["nuc_median_pearson"] < 0.5]

print("=== Performance tiers (Nuc Median Pearson r) ===")
print(f"Strong   (r ≥ 0.9,  n={len(strong):2d}): {', '.join(strong['target'])}")
print(f"Good     (0.8–0.9,  n={len(good):2d}): {', '.join(good['target'])}")
print(f"Moderate (0.7–0.8,  n={len(moderate):2d}): {', '.join(moderate['target'])}")
print(f"Weak     (0.5–0.7,  n={len(weak):2d}): {', '.join(weak['target'])}")
print(f"Failed   (r < 0.5,  n={len(failed):2d}): {', '.join(failed['target']) if len(failed) else 'none'}")

print()
print("=== Binary group summary ===")
print(f"{'Group':<22} {'n':>3}  {'mean':>6}  {'median':>7}  {'min':>5}  {'max':>5}")
print("-" * 55)
for g in ["cell_cycle", "non_cell_cycle"]:
    vals = df[df["group"] == g]["nuc_median_pearson"].values
    print(f"{GROUP_LABELS[g]:<22} {len(vals):>3}  {np.mean(vals):>6.3f}  {np.median(vals):>7.3f}  {np.min(vals):>5.3f}  {np.max(vals):>5.3f}")

print()
print("=== Notable cases ===")
# Biggest SSIM vs Pearson gap (pixel-poor but cell-level useful)
df["ssim_pearson_gap"] = df["nuc_median_pearson"] - df["ssim_mean"]
top_gap = df.nlargest(3, "ssim_pearson_gap")[["target", "group", "ssim_mean", "nuc_median_pearson", "ssim_pearson_gap"]]
print("Largest Pearson–SSIM gap (pixel-poor but cell-level useful):")
print(top_gap.to_string(index=False))
print()
# Signaling proteins that beat expectations
sig = df[df["group"] == "non_cell_cycle"].nlargest(3, "nuc_median_pearson")[["target", "nuc_median_pearson", "ssim_mean"]]
print("Best-performing signaling proteins (above expectation):")
print(sig.to_string(index=False))
print()
# Worst overall
bot = df.nsmallest(3, "nuc_median_pearson")[["target", "group", "nuc_median_pearson", "ssim_mean"]]
print("Worst performers:")
print(bot.to_string(index=False))